# VacationPy
---

## Starter Code to Import Libraries and Load the Weather and Coordinates Data

In [1]:
# Dependencies and Setup
import hvplot.pandas
import pandas as pd
import requests

# Import API key
from api_keys import geoapify_key

In [2]:
# Load the CSV file created in Part 1 into a Pandas DataFrame
city_data_df = pd.read_csv("output_data/cities.csv")

# Display sample data
city_data_df.head()

,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
0,0,mora,11.0461,14.1401,23.02,22,21,1.65,CM,2025-01-13
1,1,metlili chaamba,32.2667,3.6333,9.52,46,40,3.60,DZ,2025-01-13
2,2,hassi messaoud,31.6804,6.0729,9.92,43,1,1.03,DZ,2025-01-13
3,3,ust-nera,64.5667,143.2000,-34.22,94,99,0.96,RU,2025-01-13
4,4,port mathurin,-19.6833,63.4167,26.10,79,6,7.65,MU,2025-01-13


In [3]:
# Count how many zero or negative humidity values exist
invalid_count = (city_data_df['Humidity'] <= 0).sum()
print(f"Number of zero or negative humidity values: {invalid_count}")

Number of zero or negative humidity values: 0


In [4]:
# Count how many null (NaN) values are in Humidity
null_count = city_data_df['Humidity'].isnull().sum()
print(f"Number of null (NaN) humidity values: {null_count}")

Number of null (NaN) humidity values: 0


---

### Step 1: Create a map that displays a point for every city in the `city_data_df` DataFrame. The size of the point should be the humidity in each city.

In [17]:
%%capture --no-display

# Configure the map plot
map_plot=city_data_df.hvplot.points(
    'Lat',
    'Lng',
    geo=True,
    tiles='OSM',
    color='City',
    frame_width=650,
    frame_height=500,
    size='Humidity',
)

# Display the map
map_plot

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Lat,Lng]   (City,Humidity)

### Step 2: Narrow down the `city_data_df` DataFrame to find your ideal weather condition

In [6]:
# Narrow down cities that fit criteria and drop any results with null values
ideal_weather_df=city_data_df.loc[(city_data_df['Max Temp']<=28)\
                                    &(city_data_df['Max Temp']>=0)\
                                    &(city_data_df['Cloudiness']<=90)\
                                    &(city_data_df['Humidity']<=90)\
                                    &(city_data_df['Country']!='RU')\
                                    &(city_data_df['Country']>='IT')\
                                    &(city_data_df['Country']>='RE')\
                                    &(city_data_df['Country']>='AU'), :]

# Drop any rows with null values
ideal_weather_df.dropna()

# Display sample data
ideal_weather_df.head()

,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
6,6,port-aux-francais,-49.3500,70.2167,5.08,80,70,20.28,TF,2025-01-13
25,25,ataq,14.5377,46.8319,18.32,47,0,2.24,YE,2025-01-13
31,31,jisr ash shughur,35.8143,36.3206,10.58,74,11,0.93,SY,2025-01-13
46,46,east london,-33.0153,27.9116,20.93,87,4,7.37,ZA,2025-01-13
57,57,kruisfontein,-34.0033,24.7314,19.41,77,0,3.08,ZA,2025-01-13


### Step 3: Create a new DataFrame called `hotel_df`.

In [7]:
# Use the Pandas copy function to create DataFrame called hotel_df to store the city, country, coordinates, and humidity
hotel_df=ideal_weather_df.iloc[:, [1,8,2,3,5]].copy()

# Add an empty column, "Hotel Name," to the DataFrame so you can store the hotel found using the Geoapify API
hotel_df["Hotel Name"]= ''

# Display sample data
hotel_df.head()

,City,Country,Lat,Lng,Humidity,Hotel Name
6,port-aux-francais,TF,-49.3500,70.2167,80,
25,ataq,YE,14.5377,46.8319,47,
31,jisr ash shughur,SY,35.8143,36.3206,74,
46,east london,ZA,-33.0153,27.9116,87,
57,kruisfontein,ZA,-34.0033,24.7314,77,


### Step 4: For each city, use the Geoapify API to find the first hotel located within 10,000 metres of your coordinates.

In [13]:
print(requests.Request('GET', base_url, params=params).prepare().url)

https://api.geoapify.com/v2/places?categories=accommodation.hotel&bias=proximity%3A23.5654%2C119.5863&apiKey=784156b606cf4d70add8130e0dabcae4&limit=10&filter=circle%3A23.5654%2C+119.5863%2C+10000


In [15]:
# Set parameters to search for a hotel
radius = 10000
params = {
    'categories':'accommodation.hotel',
    'apiKey': geoapify_key,
    'limit': 10
    }

# Print a message to follow up the hotel search
print("Starting hotel search")

# Iterate through the hotel_df DataFrame
for index, row in hotel_df.iterrows():
    # get latitude, longitude from the DataFrame
    lat=row['Lat']
    lon=row['Lng']

    # Add the current city's latitude and longitude to the params dictionary
    params["filter"] = f'circle:{lon},{lat},{radius}'
    params["bias"] = f'proximity:{lon},{lat}'

    # Set base URL
    base_url = "https://api.geoapify.com/v2/places"

    # Make and API request using the params dictionary
    name_address = requests.get(base_url, params=params)

    # Convert the API response to JSON format
    name_address = name_address.json()

    # Grab the first hotel from the results and store the name in the hotel_df DataFrame
    try:
        hotel_df.loc[index, "Hotel Name"] = name_address["features"][0]["properties"]["name"]
    except (KeyError, IndexError):
        # If no hotel is found, set the hotel name as "No hotel found".
        hotel_df.loc[index, "Hotel Name"] = "No hotel found"

    # Log the search results
    print(f"{hotel_df.loc[index, 'City']} - nearest hotel: {hotel_df.loc[index, 'Hotel Name']}")

# Display sample data
hotel_df

Starting hotel search
port-aux-francais - nearest hotel: Keravel
ataq - nearest hotel: الشارقة بلازا
jisr ash shughur - nearest hotel: No hotel found
east london - nearest hotel: No hotel found
kruisfontein - nearest hotel: No hotel found
mtambile - nearest hotel: SMZ Hoteli Ya Mkoani
fort bragg - nearest hotel: Airborne Inn Lodging
bel ombre - nearest hotel: Villa La Cachette
atamyrat - nearest hotel: hotel housing working
kill devil hills - nearest hotel: Mariner Days Inn & Suites
buka - nearest hotel: No hotel found
beaufort west - nearest hotel: Matoppo Inn
broome - nearest hotel: No hotel found
buala - nearest hotel: No hotel found
port alfred - nearest hotel: No hotel found
newman - nearest hotel: No hotel found
lompoc - nearest hotel: Red Roof Inn Lompoc
saint-philippe - nearest hotel: Le Baril
al ghayzah - nearest hotel: فندق تاج العرب
hanceville - nearest hotel: No hotel found
la'ie - nearest hotel: No hotel found
san patricio - nearest hotel: No hotel found
badger - nearest h

,City,Country,Lat,Lng,Humidity,Hotel Name
6,port-aux-francais,TF,-49.3500,70.2167,80,Keravel
25,ataq,YE,14.5377,46.8319,47,الشارقة بلازا
31,jisr ash shughur,SY,35.8143,36.3206,74,No hotel found
46,east london,ZA,-33.0153,27.9116,87,No hotel found
57,kruisfontein,ZA,-34.0033,24.7314,77,No hotel found
92,mtambile,TZ,-5.3833,39.7000,80,SMZ Hoteli Ya Mkoani
113,fort bragg,US,35.1390,-79.0060,41,Airborne Inn Lodging
116,bel ombre,SC,-4.6167,55.4167,71,Villa La Cachette
122,atamyrat,TM,37.8357,65.2106,42,hotel housing working
125,kill devil hills,US,36.0307,-75.6760,52,Mariner Days Inn & Suites


### Step 5: Add the hotel name and the country as additional information in the hover message for each city in the map.

In [16]:
%%capture --no-display

# Configure the map plot
hotel_plot=hotel_df.hvplot.points(
    "Lng",
    "Lat",
    geo=True,
    tiles="OSM",
    frame_width=650,
    frame_height=500,
    size="Humidity",
    color="City",
    hover_cols=["Hotel Name", "Country"]
)

# Display the map
hotel_plot

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Lng,Lat]   (City,Humidity,Hotel Name,Country)